In [34]:

import os
import sys
import json
import numpy as np
import pandas as pd


#BASE_DIR      = os.path.dirname(os.path.dirname(os.path.abspath(current_dir)))

current_dir=os.getcwd()

BASE_DIR      = os.path.dirname(os.path.abspath(current_dir))
#
BRONZE_DIR    = os.path.join(BASE_DIR, "data", "Bronze layer")
SILVER_DIR    = os.path.join(BASE_DIR, "data", "Silver layer")
GOLD_DIR      = os.path.join(BASE_DIR, "data", "Gold layer")

ARTIFACTS_DIR = os.path.join(BASE_DIR, "main_scripts", "pipeline_artifacts")

SCALER_PATH  = os.path.join(ARTIFACTS_DIR, "scaler_params.json")
ENCODER_PATH = os.path.join(ARTIFACTS_DIR, "encoder_params.json")


OUT_DIR       = os.path.join(BASE_DIR, "data","model_output")
os.makedirs(OUT_DIR, exist_ok=True)


In [35]:
OUT_DIR

'/Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/data/model_output'

In [36]:
import pickle

In [37]:
def load_data(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Input file not found: {path}")
    df = pd.read_csv(path)
    print(f"  Loaded {os.path.basename(path)}: {df.shape}")
    return df


def silver_output_path(input_path: str) -> str:
    name = os.path.basename(input_path)
    stem = os.path.splitext(name)[0]
    return os.path.join(SILVER_DIR, f"{stem}_cleaned.csv")

In [38]:
def gold_output_path(input_path: str) -> str:
    name = os.path.basename(input_path)          # train_cleaned.csv
    stem = name.replace("_cleaned", "")           # train.csv
    stem = os.path.splitext(stem)[0]              # train
    return os.path.join(GOLD_DIR, f"{stem}_engineered.csv")


In [39]:
def save_artifacts(scaler_params: dict, encoder_params: dict) -> None:
    os.makedirs(ARTIFACTS_DIR, exist_ok=True)
    with open(SCALER_PATH, "w") as f:
        json.dump(scaler_params, f, indent=2)
    with open(ENCODER_PATH, "w") as f:
        json.dump(encoder_params, f, indent=2)
    print(f"  Artifacts saved → {ARTIFACTS_DIR}/")


def load_artifacts() -> tuple[dict, dict]:
    missing = [p for p in (SCALER_PATH, ENCODER_PATH) if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Pipeline artifacts are missing — the training pipeline must be run first "
            "before applying inference transformations.\n"
            f"Missing files:\n" + "\n".join(f"  {p}" for p in missing)
        )
    with open(SCALER_PATH) as f:
        scaler_params = json.load(f)
    with open(ENCODER_PATH) as f:
        encoder_params = json.load(f)
    return scaler_params, encoder_params


def _apply_standard_scaler(df: pd.DataFrame, scaler_params: dict) -> pd.DataFrame:
    for col, stats in scaler_params.items():
        if col not in df.columns:
            raise ValueError(f"Expected numeric column '{col}' not found in input.")
        df[col] = (df[col] - stats["mean"]) / stats["std"]
    return df


def _apply_encoders(df: pd.DataFrame, encoder_params: dict, BINARY_COLS :dict ,ORDINAL_COLS :dict , CATEGORICAL_COLS:list) -> pd.DataFrame:
    # Fixed binary mappings
    for col, mapping in BINARY_COLS.items():
        if col not in df.columns:
            raise ValueError(f"Expected binary column '{col}' not found in input.")
        unseen = set(df[col].dropna().unique()) - set(mapping)
        if unseen:
            print(f"  [WARN] '{col}': unseen values {unseen} → mapped to -1")
        df[col] = df[col].map(mapping).fillna(-1).astype(int)

    # Fixed ordinal mappings
    for col, mapping in ORDINAL_COLS.items():
        if col not in df.columns:
            raise ValueError(f"Expected ordinal column '{col}' not found in input.")
        unseen = set(df[col].dropna().unique()) - set(mapping)
        if unseen:
            print(f"  [WARN] '{col}': unseen values {unseen} → mapped to -1")
        df[col] = df[col].map(mapping).fillna(-1).astype(int)

    # Learned categorical mappings (from train)
    for col in CATEGORICAL_COLS:
        if col not in df.columns:
            raise ValueError(f"Expected categorical column '{col}' not found in input.")
        mapping = encoder_params.get(col, {})
        unseen = set(df[col].dropna().unique()) - set(mapping)
        if unseen:
            print(f"  [WARN] '{col}': unseen categories {unseen} → mapped to -1")
        df[col] = df[col].map(mapping).fillna(-1).astype(int)

    return df
def to_z(scaler,col, raw_value):
    mean = scaler[col]["mean"]
    std  = scaler[col]["std"]
    return (raw_value - mean) / std


In [40]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. INFERENCE PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def run_inference_pipeline(df: pd.DataFrame, output_path: str, BINARY_COLS :dict ,ORDINAL_COLS :dict , CATEGORICAL_COLS:list) -> pd.DataFrame:
    print("  [inference] Loading artifacts …")
    scaler_params, encoder_params = load_artifacts()

    # Drop id; no target present
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
    if TARGET_RAW in df.columns:
        print(f"  [inference] Warning: '{TARGET_RAW}' found in test file — dropping it.")
        df = df.drop(columns=[TARGET_RAW])

    df["income_exact"]  = np.floor(df["Annual_Income_USD"]).astype(int).astype(str)
    df["income_100"]    = (np.floor(df["Annual_Income_USD"] / 100) * 100).astype(int).astype(str)
    df["income_1000"]   = (np.floor(df["Annual_Income_USD"] / 1000) * 1000).astype(int).astype(str)

    df["Daily_Commute_km_exact"]  = np.floor(df["Daily_Commute_km"]).astype(int).astype(str)
#df["Age_exact"]  = np.floor(df["Age"]).astype(int).astype(str)

    # Apply same transformations fitted on train
    df = _apply_standard_scaler(df, scaler_params)
    df = _apply_encoders(df, encoder_params,BINARY_COLS ,ORDINAL_COLS , CATEGORICAL_COLS)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    df.to_csv(output_path, index=False)
    print(f"  [inference] Silver layer saved → {output_path}  {df.shape}")
    return df


## Inputs

In [41]:
sel_set="test"
#sel_set="train"


train_set=os.path.join(BRONZE_DIR, "train.csv")
test_set=os.path.join(BRONZE_DIR, "test.csv")

INPUT_FILES = [
    train_set,
    test_set,
]
if sel_set=="train":
    selected_set=train_set
elif sel_set=="test":
    selected_set=test_set


# Main

## Data cleaning 

In [42]:
df_test=load_data(selected_set)

  Loaded test.csv: (286571, 14)


In [43]:
output = silver_output_path(selected_set)

DROP_COLS   = [] #["id"]
TARGET_RAW  = "Will_Buy_EV"
TARGET_OUT  = "will_buy_ev"


NUMERIC_COLS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

CATEGORICAL_COLS = ["Gender", "City_Type", "Current_Car_Type"]

# Define transformations
BINARY_COLS = {
    "Home_Charging_Possible": {"No": 0, "Yes": 1},
    "Subsidy_Available":      {"No": 0, "Yes": 1},
}

ORDINAL_COLS = {
    "Range_Anxiety_Level": {"Low": 0, "Medium": 1, "High": 2},
}

In [44]:
df_test_procesed=run_inference_pipeline(df_test, output,BINARY_COLS ,ORDINAL_COLS , CATEGORICAL_COLS)

  [inference] Loading artifacts …
  [inference] Silver layer saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/data/Silver layer/test_cleaned.csv  (286571, 18)


In [45]:
df_test_procesed

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,income_exact,income_100,income_1000,Daily_Commute_km_exact
0,668665,1.084299,-0.594955,-0.814625,0.394055,0.519398,-0.612405,0.744881,1,1,2,1,0,0,67725,67700,67000,16
1,668666,-0.391379,2.375933,0.520099,0.394055,1.028713,0.351613,0.744881,1,2,1,0,0,0,152835,152800,152000,41
2,668667,1.627970,0.073573,1.128734,-0.977170,1.283371,0.737221,0.744881,0,2,2,0,0,0,86877,86800,86000,53
3,668668,-0.624380,-1.325581,0.103666,0.394055,-0.244575,0.158810,0.744881,0,1,2,1,0,0,46794,46700,46000,34
4,668669,0.618296,0.956532,1.336951,-0.977170,0.264740,-0.612405,-1.354316,0,1,2,1,1,0,112172,112100,112000,57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286566,955231,1.239634,-0.312038,-0.440902,-0.977170,-0.499233,-0.805209,-1.354316,1,1,1,1,0,0,75830,75800,75000,23
286567,955232,-1.556387,-0.181732,1.230173,0.394055,0.519398,-0.419601,0.045149,1,1,2,1,1,0,79563,79500,79000,55
286568,955233,-1.323386,-1.277096,0.557472,1.765280,0.519398,-0.033994,0.744881,0,1,2,1,0,0,48183,48100,48000,42
286569,955234,-1.168051,0.805492,-1.449954,0.394055,2.302002,-0.033994,-1.354316,0,2,1,0,0,0,107845,107800,107000,5


In [46]:
df_test

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
0,668665,61,67725.0,16.9,2,7,4,4.0,Male,Suburban,Sedan,Yes,No,Low
1,668666,42,152835.0,41.9,2,9,9,4.0,Male,Urban,SUV,No,No,Low
2,668667,68,86877.0,53.3,1,10,11,4.0,Female,Urban,Sedan,No,No,Low
3,668668,39,46794.0,34.1,2,4,8,4.0,Female,Suburban,Sedan,Yes,No,Low
4,668669,55,112172.0,57.2,1,6,4,1.0,Female,Suburban,Sedan,Yes,Yes,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286566,955231,63,75830.0,23.9,1,3,3,1.0,Male,Suburban,SUV,Yes,No,Low
286567,955232,27,79563.0,55.2,2,7,5,3.0,Male,Suburban,Sedan,Yes,Yes,Low
286568,955233,30,48183.0,42.6,3,7,7,4.0,Female,Suburban,Sedan,Yes,No,Low
286569,955234,32,107845.0,5.0,2,14,7,1.0,Female,Urban,SUV,No,No,Low


## Gold Layer: Interactions

In [47]:
with open("pipeline_artifacts/scaler_params.json") as f:
    scaler = json.load(f)

In [48]:
# Income thresholds
# ── Thresholds on z-score scale (pre-computed from scaler_params.json) ────────
# Raw source → z = (raw - mean) / std
# Income (mean=84769.27, std=28648.01): $68k→-0.5854  $109k→0.8476  $167k→2.8636
INC_BINS_Z = [-np.inf,
              to_z(scaler,"Annual_Income_USD", 68000),
              to_z(scaler,"Annual_Income_USD", 109000),
              to_z(scaler,"Annual_Income_USD", 167000),
              np.inf]
INC_LABELS  = [0, 1, 2, 3]   # ordinal: 0=Low, 1=Mid, 2=High, 3=VeryHigh

# Commute thresholds
# Commute (mean=32.16, std=18.73): 10km→-1.1830  23km→-0.4890  59km→1.4331

COMMUTE_SWEET_LOW_Z  = to_z(scaler,"Daily_Commute_km", 10)
COMMUTE_SWEET_HIGH_Z = to_z(scaler,"Daily_Commute_km", 23)
COMMUTE_LONG_Z       = to_z(scaler,"Daily_Commute_km", 59)

# Env thresholds
# Env Concern (mean=2.94, std=1.43): raw ≥4 → z≥0.7449  raw ≤2 → z≤-0.6546

ENV_HIGH_Z = to_z(scaler,"Environmental_Concern_Level", 4)
ENV_LOW_Z  = to_z(scaler,"Environmental_Concern_Level", 2)


# Silver encoding reference (from encoder_params.json / fixed mappings):
# Subsidy_Available:    No=0, Yes=1
# Range_Anxiety_Level:  Low=0, Medium=1, High=2


In [49]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add 10 interaction features to the Silver DataFrame in-place copy."""
    out = df.copy()

    env     = df["Environmental_Concern_Level"]   # z-scored
    income  = df["Annual_Income_USD"]              # z-scored
    commute = df["Daily_Commute_km"]               # z-scored
    subsidy = df["Subsidy_Available"]              # 0=No, 1=Yes
    anxiety = df["Range_Anxiety_Level"]            # 0=Low, 1=Medium, 2=High

    # Income segment (ordinal 0–3) — from DT analysis thresholds
    income_seg = pd.cut(income, bins=INC_BINS_Z, labels=INC_LABELS, right=False).astype(int)

    # ── Binary flags ──────────────────────────────────────────────────────────
    out["flag_env_high_x_subsidy"]       = ((env >= ENV_HIGH_Z)  & (subsidy == 1)).astype(int)
    out["flag_env_low_no_subsidy"]       = ((env <= ENV_LOW_Z)   & (subsidy == 0)).astype(int)
    out["flag_subsidy_x_low_anxiety"]    = ((subsidy == 1)        & (anxiety == 0)).astype(int)
    out["flag_high_concern_low_anxiety"] = ((env >= ENV_HIGH_Z)  & (anxiety != 2)).astype(int)
    out["flag_commute_sweet_spot"]       = ((commute >= COMMUTE_SWEET_LOW_Z) & (commute < COMMUTE_SWEET_HIGH_Z)).astype(int)
    out["flag_commute_long_haul"]        = (commute > COMMUTE_LONG_Z).astype(int)

    # ── Segment composites ────────────────────────────────────────────────────
    out["income_segment"]   = income_seg
    # Encodes the full income × anxiety grid as a single integer (0–11)
    out["income_x_anxiety"] = income_seg * 3 + anxiety

    # ── Numeric interaction features ──────────────────────────────────────────
    # income_segment+1 gives 1–4 (same scale as quartile without data-leakage)
    out["env_x_income_product"] = env * (income_seg + 1)
    # env is already z-scored; adding subsidy (0/1) shifts the distribution for Yes
    out["ev_propensity_score"]  = env + subsidy.astype(float)

    return out

In [50]:
selected_set[-8:]

'test.csv'

In [51]:
train_set_silver=os.path.join(SILVER_DIR, "train_cleaned.csv")
test_set_silver=os.path.join(SILVER_DIR, "test_cleaned.csv")
if sel_set=='test':
    path=test_set_silver
elif sel_set=='train':
    path=train_set_silver
output = gold_output_path(path)


In [52]:
silver_cols = list(pd.read_csv(train_set_silver, nrows=0).columns)
df = load_data(path)


  Loaded test_cleaned.csv: (286571, 18)


In [53]:
df

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,income_exact,income_100,income_1000,Daily_Commute_km_exact
0,668665,1.084299,-0.594955,-0.814625,0.394055,0.519398,-0.612405,0.744881,1,1,2,1,0,0,67725,67700,67000,16
1,668666,-0.391379,2.375933,0.520099,0.394055,1.028713,0.351613,0.744881,1,2,1,0,0,0,152835,152800,152000,41
2,668667,1.627970,0.073573,1.128734,-0.977170,1.283371,0.737221,0.744881,0,2,2,0,0,0,86877,86800,86000,53
3,668668,-0.624380,-1.325581,0.103666,0.394055,-0.244575,0.158810,0.744881,0,1,2,1,0,0,46794,46700,46000,34
4,668669,0.618296,0.956532,1.336951,-0.977170,0.264740,-0.612405,-1.354316,0,1,2,1,1,0,112172,112100,112000,57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286566,955231,1.239634,-0.312038,-0.440902,-0.977170,-0.499233,-0.805209,-1.354316,1,1,1,1,0,0,75830,75800,75000,23
286567,955232,-1.556387,-0.181732,1.230173,0.394055,0.519398,-0.419601,0.045149,1,1,2,1,1,0,79563,79500,79000,55
286568,955233,-1.323386,-1.277096,0.557472,1.765280,0.519398,-0.033994,0.744881,0,1,2,1,0,0,48183,48100,48000,42
286569,955234,-1.168051,0.805492,-1.449954,0.394055,2.302002,-0.033994,-1.354316,0,2,1,0,0,0,107845,107800,107000,5


In [54]:
df = engineer_features(df)


In [55]:



new_cols = [c for c in df.columns if c not in silver_cols]
print(f"  Engineered features ({len(new_cols)}): {new_cols}")


os.makedirs(GOLD_DIR, exist_ok=True)
df.to_csv(output, index=False)
print(f"  Gold layer saved → {output}  {df.shape}")

  Engineered features (11): ['id', 'flag_env_high_x_subsidy', 'flag_env_low_no_subsidy', 'flag_subsidy_x_low_anxiety', 'flag_high_concern_low_anxiety', 'flag_commute_sweet_spot', 'flag_commute_long_haul', 'income_segment', 'income_x_anxiety', 'env_x_income_product', 'ev_propensity_score']
  Gold layer saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/data/Gold layer/test_engineered.csv  (286571, 28)


In [56]:
df

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,...,flag_env_high_x_subsidy,flag_env_low_no_subsidy,flag_subsidy_x_low_anxiety,flag_high_concern_low_anxiety,flag_commute_sweet_spot,flag_commute_long_haul,income_segment,income_x_anxiety,env_x_income_product,ev_propensity_score
0,668665,1.084299,-0.594955,-0.814625,0.394055,0.519398,-0.612405,0.744881,1,1,...,0,0,0,1,1,0,0,0,0.744881,0.744881
1,668666,-0.391379,2.375933,0.520099,0.394055,1.028713,0.351613,0.744881,1,2,...,0,0,0,1,0,0,2,6,2.234643,0.744881
2,668667,1.627970,0.073573,1.128734,-0.977170,1.283371,0.737221,0.744881,0,2,...,0,0,0,1,0,0,1,3,1.489762,0.744881
3,668668,-0.624380,-1.325581,0.103666,0.394055,-0.244575,0.158810,0.744881,0,1,...,0,0,0,1,0,0,0,0,0.744881,0.744881
4,668669,0.618296,0.956532,1.336951,-0.977170,0.264740,-0.612405,-1.354316,0,1,...,0,0,1,0,0,0,2,6,-4.062949,-0.354316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286566,955231,1.239634,-0.312038,-0.440902,-0.977170,-0.499233,-0.805209,-1.354316,1,1,...,0,1,0,0,0,0,1,3,-2.708633,-1.354316
286567,955232,-1.556387,-0.181732,1.230173,0.394055,0.519398,-0.419601,0.045149,1,1,...,0,0,1,0,0,0,1,3,0.090297,1.045149
286568,955233,-1.323386,-1.277096,0.557472,1.765280,0.519398,-0.033994,0.744881,0,1,...,0,0,0,1,0,0,0,0,0.744881,0.744881
286569,955234,-1.168051,0.805492,-1.449954,0.394055,2.302002,-0.033994,-1.354316,0,2,...,0,1,0,0,0,0,1,3,-2.708633,-1.354316


## Model prediction

In [57]:
# Load model
with open("pipeline_artifacts/best_model.pkl", "rb") as f:
    model = pickle.load(f)

# Open and load the JSON file
with open('pipeline_artifacts/model_metadata.json', 'r') as file:
    model_metadata = json.load(file)

model_name=model_metadata["model_name"]
feature_set=model_metadata["feature_set"]
model_col=list(model.feature_names_in_)


In [58]:
model_col

[np.str_('Age'),
 np.str_('Annual_Income_USD'),
 np.str_('Daily_Commute_km'),
 np.str_('Number_of_Cars_Owned'),
 np.str_('Charging_Stations_Near_Home'),
 np.str_('Charging_Stations_Near_Work'),
 np.str_('Environmental_Concern_Level'),
 np.str_('Gender'),
 np.str_('City_Type'),
 np.str_('Current_Car_Type'),
 np.str_('Home_Charging_Possible'),
 np.str_('Subsidy_Available'),
 np.str_('Range_Anxiety_Level'),
 np.str_('income_exact'),
 np.str_('income_100'),
 np.str_('income_1000'),
 np.str_('Daily_Commute_km_exact'),
 np.str_('flag_env_high_x_subsidy'),
 np.str_('flag_env_low_no_subsidy'),
 np.str_('flag_subsidy_x_low_anxiety'),
 np.str_('flag_high_concern_low_anxiety'),
 np.str_('flag_commute_sweet_spot'),
 np.str_('flag_commute_long_haul'),
 np.str_('income_segment'),
 np.str_('income_x_anxiety'),
 np.str_('env_x_income_product'),
 np.str_('ev_propensity_score')]

In [59]:
X = df[model_col]
# Predict probabilities
probabilities = model.predict_proba(X)[:, 1]

# Predict classes (0/1) at default 0.5 threshold
classes = model.predict(X)

In [60]:
submission = pd.DataFrame({
    "id":          df["id"],
    "Will_Buy_EV": probabilities,
})

In [61]:
submission

,id,Will_Buy_EV
0,668665,0.083176
1,668666,0.081078
2,668667,0.019007
3,668668,0.013476
4,668669,0.070206
...,...,...
286566,955231,0.000470
286567,955232,0.683959
286568,955233,0.005236
286569,955234,0.000238


In [62]:
filename=f"{sel_set}_{model_name}_output.csv"
SUBMISSION_PATH = os.path.join(OUT_DIR, filename)


In [63]:
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\n  Submission saved → {SUBMISSION_PATH}  ({len(submission):,} rows)")


  Submission saved → /Users/victor.loaiza/Documents/Projects /Predicting Electric Vehicle Purchases/Manual/data/model_output/test_XGBoost_output.csv  (286,571 rows)


In [64]:
submission["Will_Buy_EV"].mean()

np.float32(0.29107726)

In [65]:
#np.float32(0.29238105) train